# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuneezaKhan01/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule: Prioritize pages with higher CTR gap for review.
Reason code: high_ctr_gap
Action: review

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("content_refresh_anonymized.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
d = df[df["clicks_prev_30d"] >= 5].copy()
d["declined"] = (d["clicks_last_30d"] < d["clicks_prev_30d"]).astype(int)

d["ctr_prev"] = d["clicks_prev_30d"] / d["impressions_prev_30d"].replace(0, float("nan"))
d = d.dropna(subset=["ctr_prev"])
d["ctr_gap"] = d["ctr_prev"] - d.groupby("position_tier")["ctr_prev"].transform("median")

print("pages kept:", len(d), "| decline rate:", round(d["declined"].mean(), 3))
d["score"] = d["ctr_gap"]
d = d.sort_values("score", ascending=False)
d["rank"] = range(1, len(d) + 1)
d["reason_code"] = "high_ctr_gap"
d["action"] = "review"
output = d[["rank", "score", "reason_code", "action"]].copy()

output.to_csv("baseline_action_score.csv", index=False)

print("Saved:", output.shape)
output.head(10)
import os

os.makedirs("work/outputs", exist_ok=True)

output.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved:", "work/outputs/baseline_action_score.csv")

pages kept: 5019 | decline rate: 0.63
Saved: (5019, 4)
Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def bucket_table(data, col, outcome="declined", q=5):
    b = pd.qcut(data[col], q, duplicates="drop")
    t = data.groupby(b, observed=True)[outcome].agg(n="size", decline_rate="mean").round(3)
    print(f"--- {col} ---"); print(t); return t

bucket_table(d, "days_since_last_update")
bucket_table(d, "ctr_gap")
top20 = d.head(20).copy()

print(top20[[
    "rank",
    "score",
    "days_since_last_update",
    "ctr_gap",
    "clicks_prev_30d",
    "impressions_prev_30d",
    "reason_code",
    "action"
]].to_string(index=False))
top20["confidence_note"] = "High CTR gap based on the baseline rule."
top20["what_would_make_it_wrong"] = "The CTR gap may not reflect a real content problem."

print(top20[[
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]].to_string(index=False))


--- days_since_last_update ---
                           n  decline_rate
days_since_last_update                    
(4.999, 20.0]           2214         0.657
(20.0, 104.0]           2773         0.606
(104.0, 211.0]            32         0.812
--- ctr_gap ---
                          n  decline_rate
ctr_gap                                  
(-0.00717, -0.00186]   1004         0.505
(-0.00186, -0.000681]  1004         0.604
(-0.000681, 0.000797]  1003         0.636
(0.000797, 0.00358]    1004         0.661
(0.00358, 0.156]       1004         0.743
 rank    score  days_since_last_update  ctr_gap  clicks_prev_30d  impressions_prev_30d  reason_code action
    1 0.156259                      20 0.156259                8                    50 high_ctr_gap review
    2 0.130588                      20 0.130588                9                    67 high_ctr_gap review
    3 0.064021                      20 0.064021               33                   487 high_ctr_gap review
    4 0.058206  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
**Weak picks:**  
**Weak picks:**  
Some top-ranked pages have very few historical impressions, such as ranks 1, 2, 4, 5, and 8. Their high CTR gap may be noisy because it is based on a small number of observations.

**Leakage check:**  
The score uses only `ctr_gap`, which is calculated from historical `clicks_prev_30d`, `impressions_prev_30d`, and `position_tier`. Future clicks and the `declined` outcome were not used to calculate the score. No product flags were used in the score.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak = top20.nsmallest(5, "impressions_prev_30d")

print("Weak picks:")
print(weak[[
    "rank",
    "score",
    "clicks_prev_30d",
    "impressions_prev_30d"
]].to_string(index=False))

print("\nLeakage check:")
print("Score uses: ctr_gap only")
print("ctr_gap uses: clicks_prev_30d, impressions_prev_30d, position_tier")
print("Future clicks and decline label were NOT used in the score.")


Weak picks:
 rank    score  clicks_prev_30d  impressions_prev_30d
    1 0.156259                8                    50
    2 0.130588                9                    67
    4 0.058206                7                   113
    8 0.038276                5                   119
    5 0.043878                6                   126

Leakage check:
Score uses: ctr_gap only
ctr_gap uses: clicks_prev_30d, impressions_prev_30d, position_tier
Future clicks and decline label were NOT used in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.